## Metrics on Kilonovae

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
import healpy as hp

In [ ]:
import rubin_sim.maf as maf

# import rubin_sim.utils as rsUtils
import rubin_scheduler.utils as rsUtils
from rubin_sim.data import get_baseline

The kilonovae discovery metric is similar to some other population-based metrics (such as the microlensing and TDE detection metrics). The key elements are a metric -- which defines the detection or characterization criteria for each event -- and a custom slicer -- which defines the locations and lightcurves for the events. 

The slicer is a UserPointsSlicer, with additional information tied to each slicePoint. A UserSlicePoint lets the user define RA/Dec values for the slicePoints over the sky (the locations of each event). This is then augmented by additional information on *when* the events happens and what the lightcurve looks like (or any other information about each event), tied to each slicePoint. Each slicePoint then represents each event. The slicer is set up by a function related to the metric called something like `generateKNPopSlicer` (or `generateMicrolensingSlicer`) -- this function read information about the lightcurves and where they should be distributed, sets up the UserPointSlicer with that RA/Dec information, and then adds lightcurve information as appropriate.

The metric then gets the information about the observations at each slicePoint (each RA/Dec value) as well as the slicePoint information related to the lightcurve (such as the time of the peak or the shape of the event, etc) -- the brightness at the time of each observation is generally up to the metric to calculate or interpolate as it will depend on the timing of the observations from the simulation. The metric evaluates the observed points on the lightcurve against its criteria for discovery or characterization, etc. and returns an appropriate value. For some metrics this will be a simple 0/1 (discovered or not) but for other metrics it might be something like how accurately the lightcurve could be fit. 



## 1. Set up and run the KNe metric. ## 

In [ ]:
# RUBIN_SIM_DATA_DIR points to the local cache of rubin_sim/rubin_scheduler auxiliary data
# (opsim databases, dust maps, SN gamma/noise files, throughputs, ...).
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"
path_topdir = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"path_topdir = {path_topdir}")

In [ ]:
baseline_file = get_baseline()
opsim = os.path.basename(baseline_file).replace(".db", "")

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    # Scratch directory for MAF's results database and any output products from this notebook.
    data_dir_itself = tempfile.TemporaryDirectory(prefix="01_maf_KneMetrics_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

In [ ]:
# Set up MAF output
# ResultsDb is MAF's bookkeeping database that tracks which metric bundles were run/plotted here.
out_dir = data_dir
resultsDb = maf.db.ResultsDb(out_dir=out_dir)

## 2.0 Generate slicer and Metrics for KN

In [ ]:
n_events = 30000

# Kilonova parameters - set up to run on a particular subset of the KNe models
# Just one GW170817 - like model:
inj_params_list = [
    {"mej_dyn": 0.005, "mej_wind": 0.050, "phi": 30, "theta": 25.8},
]
filename = maf.get_kne_filename(inj_params_list)
slicer_single = maf.generate_kn_pop_slicer(n_events=n_events, n_files=len(filename), d_min=10, d_max=600)
metric_single = maf.KNePopMetric(output_lc=False, file_list=filename, metric_name="KNePopMetric_single")

In [ ]:
n_events = 30000

# Kilonova parameters - set up to run on a particular subset of the KNe models
# All models
filename = maf.get_kne_filename(None)
slicer_all = maf.generate_kn_pop_slicer(n_events=n_events, n_files=len(filename), d_min=10, d_max=600)
metric_all = maf.KNePopMetric(output_lc=False, file_list=filename, metric_name="KNePopMetric_all")

In [ ]:
# These summary metrics are now available as 'batches.lightcurveSummary'
# (i.e. summaryMetrics = maf.lightcurveSummary())
summaryMetrics = [
    maf.SumMetric(metric_name="Total detected"),
    maf.CountMetric(metric_name="Total lightcurves in footprint"),
    maf.CountMetric(metric_name="Total lightcurves on sky", mask_val=0),
    maf.MeanMetric(metric_name="Fraction detected in footprint"),
    maf.MeanMetric(mask_val=0, metric_name="Fraction detected of total"),
]

## 3.0 Create Bi-bundle

In [ ]:
bundle_single = maf.MetricBundle(
    metric_single,
    slicer_single,
    None,
    info_label="single model",
    run_name=opsim,
    summary_metrics=summaryMetrics,
)
bundle_all = maf.MetricBundle(
    metric_all, slicer_all, None, info_label="all models", run_name=opsim, summary_metrics=summaryMetrics
)

## 4.0 Create group bundle and run it

In [ ]:
bdict = {"single": bundle_single, "all": bundle_all}
g = maf.MetricBundleGroup(bdict, baseline_file, out_dir=out_dir)

In [ ]:
g.run_all()

In [ ]:
bdict.keys()

## 5.0 Look at the metric outputs. ##

### 5.1 The KNeMetric returns 0/1 whether the metric could be detected or not. 

In [ ]:
# If you don't want to try and plot N individual points,
plotDict = {"reduce_func": np.sum, "nside": 64, "color_min": 0}
plotFunc = maf.plots.HealpixSkyMap()
# ph = maf.plots.PlotHandler(out_dir=outDir, figformat='png', thumbnail=False)
ph = maf.plots.PlotHandler(out_dir=outDir)
for k in bdict:
    if k.startswith("KN"):
        ph.set_metric_bundles([bdict[k]])
        ph.plot(plot_func=plotFunc, plot_dicts=plotDict)

pour info :
```
bdict.keys() =
dict_keys(['single', 'all', 'KNePopMetric_single_blue_color_detect', 'KNePopMetric_single_multi_color_detect', 'KNePopMetric_single_multi_detect', 'KNePopMetric_single_red_color_detect', 'KNePopMetric_single_ztfrest_simple', 'KNePopMetric_single_ztfrest_simple_blue', 'KNePopMetric_single_ztfrest_simple_red', 'KNePopMetric_all_blue_color_detect', 'KNePopMetric_all_multi_color_detect', 'KNePopMetric_all_multi_detect', 'KNePopMetric_all_red_color_detect', 'KNePopMetric_all_ztfrest_simple', 'KNePopMetric_all_ztfrest_simple_blue', 'KNePopMetric_all_ztfrest_simple_red'])
```

In [ ]:
# If you do want to show each individual point - this is slower
plotFunc = maf.plots.BaseSkyMap()

# ph = maf.plots.PlotHandler(out_dir=outDir, figformat='png', thumbnail=False)
ph = maf.plots.PlotHandler(out_dir=out_dir)
for k in bdict:
    if k.startswith("KN"):
        ph.set_metric_bundles([bdict[k]])
        ph.plot(plot_func=plotFunc)

In [ ]:
pd.DataFrame(
    [bdict[k].summary_values for k in bdict.keys() if k.startswith("KN")],
    index=[k.replace("_", " ") for k in bdict if k.startswith("KN")],
)

In [ ]:
# Illustrate some things about the metric values: (caught in summary metrics)
m = "KNePopMetric_all_multi_detect"
print(f"How many lightcurves were added? (over the entire sky) {len(bdict[m].metric_values)}")
print(
    f"How many lightcurves were added in areas that were part of the survey footprint?",
    f"{len(bdict[m].metric_values.compressed())}",
)
print(
    f"What are the metric values for each of these light curves? "
    f"{np.unique(bdict[m].metric_values.compressed())}"
)
print(f"How many lightcurves were *successfully* detected? {bdict[m].metric_values.sum()}")
print(len(np.where(bdict[m].metric_values == 1)[0]))
frac_total = bdict[m].metric_values.sum() / len(bdict[m].metric_values)
print(f"Fraction of total lightcurves detected {frac_total}")
frac_footprint = bdict[m].metric_values.sum() / len(bdict[m].metric_values.compressed())
print(f"Fraction of lightcurves within footprint detected {frac_footprint}")

## 5.2 Look at the slicer information (how the lightcurves were added). ##

The slicer holds the slicePoint information, including the information about where the events occured and what the lightcurves looked like before adding observational information (i.e. in the model).

In [ ]:
# The *slicer* keeps the information about the injected lightcurves too
print(f"How many lightcurves added over the sky? {len(bdict[m].slicer)}")

In [ ]:
# Including their spatial, time and distance distribution
hp.mollview(
    rsUtils._healbin(
        slicer_all.slice_points["ra"],
        slicer_all.slice_points["dec"],
        slicer_all.slice_points["peak_time"],
        64,
        reduce_func=np.mean,
    ),
    unit="peak time (days)",
    title="Lightcurve Peak Times",
    min=0,
    max=3650,
)

In [ ]:
# And the distance distribution -- which we can also modify to show detected objects
distances = {}
distances["all"] = slicer_all.slice_points["distance"]
for k in bdict:
    if k.startswith("KN"):
        detected = np.where(bdict[k].metric_values == 1)
        distances[k] = distances["all"][detected]

plt.figure(figsize=(8, 5))
n, b, p = plt.hist(
    [distances[k] for k in distances],
    label=[k.replace("_", " ").replace("KNePopMetric  ", "") for k in distances],
    bins=15,
    histtype="step",
    linewidth=3,
    density=False,
    cumulative=False,
)
plt.xlabel("Luminosity distance (Mpc)", fontsize="x-large")
plt.ylabel("Recovered Kilonovae", fontsize="x-large")
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.legend(loc=(1.01, 0.4), fontsize="x-large", fancybox=True)
plt.title("Distance distribution KNe", fontsize="x-large")

In [ ]:
# And the distance distribution -- which we can also modify to show detected objects
distances = {}
distances["all"] = slicer_single.slice_points["distance"]
for k in bdict:
    if k.startswith("KN"):
        detected = np.where(bdict[k].metric_values == 1)
        distances[k] = distances["all"][detected]

plt.figure(figsize=(8, 5))
n, b, p = plt.hist(
    [distances[k] for k in distances],
    label=[k.replace("_", " ").replace("KNePopMetric  ", "") for k in distances],
    bins=15,
    histtype="step",
    linewidth=3,
    density=False,
    cumulative=False,
)
plt.xlabel("Luminosity distance (Mpc)", fontsize="x-large")
plt.ylabel("Recovered Kilonovae", fontsize="x-large")
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.legend(loc=(1.01, 0.4), fontsize="x-large", fancybox=True)
plt.title("Distance distribution KNe", fontsize="x-large")

In [ ]:
# The 'file_indx' in the slicePoint tracks which lightcurve (which file) was used for each point.
# So, with different input files we can check to see if there is a different fraction of objects detected.
indxes = np.unique(slicer_all.slice_points["file_indx"])
detfraction = np.zeros(len(indxes))
for i, indx in enumerate(indxes):
    in_indx = np.where(slicer_all.slice_points["file_indx"] == indx)[0]
    n_total = in_indx.size
    # This simply requires two detections, regardless of band
    detfraction[i] = bdict["KNePopMetric_all_multi_detect"].metric_values[in_indx].sum() / n_total

plt.figure(figsize=(13, 1))
plt.plot(indxes, detfraction, "k-")
print(
    f"Mean {np.mean(detfraction)} Min {np.min(detfraction)} Max {np.max(detfraction)} RMS {np.std(detfraction)}"
)

In [ ]:
# what do the lightcurves look like?
lcs = maf.KnLc(filename)
times = np.arange(0, 30, 0.25)

for i in range(len(filename)):
    for f in "ugrizy":
        mags = lcs.interp(times, f, lc_indx=i)
        plt.plot(times, mags)

        plt.ylim(0, -20)

### 5.3 Look at the metric outputs across the v2 runs ###


In [ ]:
families = maf.archive.get_family_descriptions()
families = families.drop(index="draft v3")
family_list = families.index.values
summaries = maf.get_metric_summaries()  # maf.get_metric_summaries(summary_source=summary_source)
metric_sets = maf.get_metric_sets()

In [ ]:
# Make a comparison of the KNePopMetric with single and all models
two_obs_detect = [
    m for m in summaries if "KNePopMetric" in m and "Total detected" in m and "multi_detect" in m
]
color_rise_detect = [
    m
    for m in summaries
    if "KNePopMetric" in m
    and "Total detected" in m
    and "ztfrest_simple" in m
    and "blue" not in m
    and "red" not in m
]
metrics = two_obs_detect + color_rise_detect
short_names = [
    "Total KNePop multi detect single model",
    "Total KNePop multi detect all models",
    "Total KNePop ztfrest_simple single model",
    "Total KNePop ztfrest_simple all models",
]
styles = ["r-", "b-"] + ["r:", "b:"]
msub = maf.create_metric_set_df("kne_comparison", metrics, short_name=short_names, style=styles)
msub

In [ ]:
these_runs = families.explode("run")["run"]
fig, ax = maf.plot_run_metric(
    summaries.loc[these_runs, two_obs_detect],
    baseline_run="baseline_v2.0_10yrs",
    metric_set=msub,
    horizontal_quantity="value",
    vertical_quantity="run",
)
fig.set_figheight(30)

In [ ]:
fig, ax = maf.plot_run_metric(
    summaries.loc[these_runs, color_rise_detect],
    baseline_run="baseline_v2.0_10yrs",
    metric_set=msub,
    horizontal_quantity="value",
    vertical_quantity="run",
)
fig.set_figheight(30)

In [ ]:
fig, ax = maf.plot_run_metric(
    summaries.loc[these_runs, msub["metric"]],
    baseline_run="baseline_v2.0_10yrs",
    metric_set=msub,
    horizontal_quantity="value",
    vertical_quantity="run",
)
fig.set_figheight(30)

In [ ]:
these_runs = families.explode("run").loc[[f for f in families.index if not f.startswith("ddf")]]["run"]
fig, ax = maf.plot_run_metric_mesh(
    summaries.loc[these_runs, msub["metric"]],
    baseline_run="baseline_v2.0_10yrs",
    metric_label_map=msub.loc["kne_comparison"]["short_name"],
)
fig.set_figwidth(20)

In [ ]:
# What works best for these metrics?
df = summaries.loc[these_runs, msub["metric"]] / summaries.loc["baseline_v2.0_10yrs", msub["metric"]]
df.iloc[np.where(df > 1.1)[0]]

So basically: 
* the old baseline (retro_baseline_v2.0_10yrs) did fairly well for simple detection, but less well for detection  + identification within the survey 
* a six-band rolling cadence concentrates visits into a short enough time period that identification works well, but due to the limited sky available at any time, does less well at detection 
* similarly with long_gaps_nightsoff0 -- the amount of new sky covered per night decreases, as well as the overall number of visits, and so identification is better but detection is poorer
* cutting down the exposure time to 20s or 22s adds enough additional visits that these bright objects are detectable 
* adding twilight NEO visits lengthens the season .. which is likely why both of these KNe metrics do better
* starting the survey at a different time of year (march_start) alters the resulting number of KNe detected .. this is basically as weather and night length sampling change, and may indicate something more about how variable the underlying metric might be in the presence of "real life" (?)
* suppressing repeat visits within the same night improves both detection and the identification of KNe. The identification increase makes sense as moving visits into adjacent nights means that the rise/fall in the objects' magnitude is easier to detect.  The simple detection improvement is likely due to just sampling a wider range of nights, giving more chances at landing an observation in a period where the KNe transient is visible.
